# 01e · 수렴 정리와 혼합: 전변동 거리, 결합, 스펙트럼 간극 (Convergence Theorem and Mixing: Total Variation, Coupling, Spectral Gap)

| 항목 | 내용 |
|---|---|
| 교재 지도 | Durrett 3e §1.5 (극한 거동, Thm 1.19 수렴 정리), §1.7 (결합 증명) · Norris §1.8 (평형으로의 수렴, Thm 1.8.3; 1.8.4 주기적 반례) · Lawler 2e §1.2 (고유값으로 본 수렴 속도 $|\lambda_2|^n$) · Ross 12e §4.4 (극한확률 vs 장기 비율) |
| 선수 노트북 | 01d, 01c |
| 예상 소요 | 100분 |
| 상태 | draft |

01d에서 정상분포 $\pi$는 **존재**했지만, 반사 랜덤 워크에서는 $P^n$의 행이 $\pi$로 **수렴하지 않았다**. 이 노트북은 그 틈을 메운다. "어디서 출발하든 $P^n(x,\cdot)\to\pi$"가 성립하는 조건(기약 + 비주기)을 **결합(coupling)** 논법으로 완전히 증명하고, 그 수렴이 **얼마나 빠른지**를 두 도구 — 전변동 거리 $d(n)$과 두 번째 고유값 $\lambda_2$ — 로 잰다. 마지막으로 비주기성을 깨서 무엇이 무너지는지, 그리고 세자로(Cesàro) 평균은 왜 살아남는지 본다.

## 0. 준비 (Setup)

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from numpy.linalg import matrix_power
from IPython.display import Markdown
from spkit import rng_for, fast, scale, setup_plots
from spkit.mc import mc_estimate, mc_proportion, compare_table, assert_close
from spkit.markov import (stationary, tv_distance, simulate_chain, ehrenfest_matrix,
                          classify_states)

SEED, rng = rng_for("01e")
setup_plots()
N_PATHS = scale(20_000)   # FAST(SPKIT_FAST=1) 모드에서는 1/10 크기
print(f"SEED={SEED}, fast={fast()}, N_PATHS={N_PATHS}")

## 1. Recall — 지난 노트북에서

책을 덮고 먼저 답을 적어 본 뒤 정답을 펼쳐 보세요.

**Q1.** (01d) 정상분포의 정의와 세 가지 계산법은?

<details><summary>정답</summary>

$\pi P=\pi$, $\sum_x\pi(x)=1$. (i) 선형계 풀이(`stationary`), (ii) $P^\top$의 고유값 1 고유벡터(`stationary_eig`), (iii) 장기 빈도 또는 $P^n$의 행(`stationary_power`, 시간 평균).

</details>

**Q2.** (01d) Ehrenfest(20)의 정상분포와 $E_{10}T_{10}$은? 일반적으로 $\pi(x)$와 $E_xT_x$의 관계는?

<details><summary>정답</summary>

$\pi=\text{Bin}(20,1/2)$, $E_{10}T_{10}=1/\pi(10)=2^{20}/\binom{20}{10}=5.675$. 기약 유한 사슬에서 $\pi(x)=1/E_xT_x$.

</details>

**Q3.** (01d) 반사 랜덤 워크에서 `stationary_power`가 실패한 이유와, 그래도 수렴하는 양은?

<details><summary>정답</summary>

주기 2라서 $P^n$의 행이 두 벡터 사이를 진동한다($P^{1000}[0]=(1/4,0,1/2,0,1/4)$, $P^{1001}[0]=(0,1/2,0,1/2,0)$). 그래도 한 경로의 시간 평균과 세자로 평균 $\frac1n\sum_{k\le n}P^k$는 $\pi=(1,2,2,2,1)/8$로 수렴한다.

</details>

**Q4.** (01c) 1단계 분석으로 $E[T_{HH}]=6$을 얻는 두 방정식은?

<details><summary>정답</summary>

$t_0=1+\frac12t_0+\frac12t_1$, $t_1=1+\frac12t_0$ ⇒ $t_1=4$, $t_0=6$. (오늘의 결합 시각 $T_c$도 같은 방식의 1단계 분석으로 다룰 수 있다.)

</details>

## 2. 이론 — 내 말로 다시 쓰기

### 2.1 정의

- **전변동 거리 (total variation distance)**: 유한 집합 $S$ 위의 확률벡터 $\mu,\nu$에 대해
  $$\|\mu-\nu\|_{TV}=\max_{A\subset S}|\mu(A)-\nu(A)|=\frac12\sum_{x\in S}|\mu(x)-\nu(x)|\qquad(\text{`tv_distance`}).$$
  두 정의가 같은 이유: 최대는 $A^*=\{x:\mu(x)>\nu(x)\}$에서 달성되고, $\mu(A^*)-\nu(A^*)=\sum_{x\in A^*}(\mu-\nu)$는 $\sum_x(\mu-\nu)=0$이므로 양의 부분과 음의 부분이 같아 $\frac12\sum|\mu-\nu|$. 값은 $[0,1]$이고, $1$이면 두 분포의 받침이 서로소. **주의**: Durrett은 계수 $\frac12$ 없이 $\sum_y|p^n(x,y)-\pi(y)|$로 쓰기도 한다 — 우리 값의 2배.
- **거리 함수** $d(n)=\max_{x}\|P^n(x,\cdot)-\pi\|_{TV}$: 최악의 출발점 기준. $\mu P$가 $\mu$에 대해 수축($\|\mu P-\nu P\|_{TV}\le\|\mu-\nu\|_{TV}$)이므로 $d(n)$은 $n$에 대해 비증가.
- **혼합시간 (mixing time)** $t_{\rm mix}(\varepsilon)=\min\{n:d(n)\le\varepsilon\}$; 관례적으로 $\varepsilon=1/4$.
- **결합 (coupling)**: 같은 확률공간 위에 두 사슬 $(X_n),(Y_n)$을 각각 **올바른 주변 법칙**(둘 다 전이행렬 $P$)으로 만들되 난수를 공유시키는 것. **결합 시각** $T_c=\min\{n:X_n=Y_n\}$; 이후 두 사슬을 붙여 놓는다($n\ge T_c$이면 $X_n=Y_n$). 오늘 시뮬레이션에서는 매 걸음 같은 균등난수 $u$를 쓰고, 증명에서는 독립으로 움직이다가 만나면 붙인다 — 둘 다 결합이다.
- **게으른(lazy) 사슬** $P_{\rm lazy}=\frac12(I+P)$: 매 걸음 동전을 던져 앞면이면 머문다. 주기성을 없애는 표준 기법이고, 고유값은 $\lambda\mapsto\frac12(1+\lambda)$로 옮겨져 모두 $\ge0$.
- **가역 사슬의 스펙트럼**: 세부균형 $\pi(x)p(x,y)=\pi(y)p(y,x)$(01f)를 만족하면 $P$는 실수 고유값 $1=\lambda_1>\lambda_2\ge\dots\ge\lambda_{|S|}\ge-1$을 갖는다. $\lambda_*=\max(|\lambda_2|,|\lambda_{|S|}|)$, **스펙트럼 간극 (spectral gap)** $\gamma_*=1-\lambda_*$.

### 2.2 정리 1 — 결합 부등식 (coupling inequality) (Durrett 3e §1.7; Norris §1.8)

> $(X_n,Y_n)$이 $X_0=x$, $Y_0\sim\pi$인 결합이고, $T_c$가 결합 시각($n\ge T_c$이면 $X_n=Y_n$)이면
> $$\|P^n(x,\cdot)-\pi\|_{TV}\le P(T_c>n).$$

**가정이 왜 필요한가**
- **$Y_n\sim\pi$ for all $n$** (정상성): $\pi$가 정상분포여야 $Y_n$의 법칙이 항상 $\pi$이고, 그래야 좌변에 $\pi$가 등장한다. $Y_0$을 다른 분포 $\nu$로 두면 부등식은 $\|\mu P^n-\nu P^n\|_{TV}\le P(T_c>n)$ — 어떤 두 초기분포에도 쓰는 일반형이다.
- **$n\ge T_c$ 이후 $X_n=Y_n$**: 결합 후 두 사슬이 같은 난수로 움직여 다시 갈라지지 않도록 구성해야 "만난 뒤에는 구별 불가"가 성립한다.

<details><summary>증명</summary>

임의의 $A\subset S$에 대해 사건을 $\{T_c\le n\}$과 $\{T_c>n\}$으로 나눈다. $\{T_c\le n\}$ 위에서는 $X_n=Y_n$이므로 $\mathbf 1\{X_n\in A\}=\mathbf 1\{Y_n\in A\}$이고, 그 부분은 차에서 소거된다:
$$P(X_n\in A)-P(Y_n\in A)=P(X_n\in A,\,T_c>n)-P(Y_n\in A,\,T_c>n)\le P(X_n\in A,\,T_c>n)\le P(T_c>n).$$
$A$와 $A^c$를 바꾸면 $P(Y_n\in A)-P(X_n\in A)\le P(T_c>n)$도 얻으므로 $|P(X_n\in A)-P(Y_n\in A)|\le P(T_c>n)$. $X_n\sim P^n(x,\cdot)$, $Y_n\sim\pi$이고 $A$가 임의이므로 TV 거리의 첫 번째 정의(집합에 대한 최대)에 의해 $\|P^n(x,\cdot)-\pi\|_{TV}\le P(T_c>n)$. $\square$

</details>

### 2.3 정리 2 — 수렴 정리 (convergence theorem) (Durrett 3e §1.5 Thm 1.19, §1.7; Norris §1.8 Thm 1.8.3)

> 유한 상태공간의 **기약·비주기** 사슬은 유일한 정상분포 $\pi$를 가지며, 모든 $x,y$에 대해 $p^n(x,y)\to\pi(y)$; 즉 $d(n)\to0$.

**가정이 왜 필요한가**
- **기약**: 아니면 극한이 출발 클래스에 의존한다(01d E3). 증명에서는 곱 사슬 $(X_n,Y_n)$의 기약성에도 쓰인다.
- **비주기**: 아니면 $p^n(x,x)$가 주기 $d$로 진동한다 — 반사 걷기와 Ehrenfest는 $\|P^n(0,\cdot)-\pi\|_{TV}=\frac12$가 영원히 유지된다(4.5절). 증명에서 비주기성이 쓰이는 **정확한 자리**는 보조정리 "$n\ge n_0$이면 $p^n(x,y)>0$"이고, 이것이 곱 사슬의 기약성을 보장한다. 주기 2이면 $(x,y)$에서 출발한 두 독립 사슬이 위상(홀짝)이 어긋나 대각선에 영원히 못 닿을 수 있다.
- **유한** (또는 양재귀): 무한 영재귀 사슬(대칭 SRW)은 기약·재귀여도 $p^n(x,y)\to0$이고 정상분포가 없다(01x). 유한성은 (i) $\pi$의 존재(01d 정리 1)와 (ii) 곱 사슬의 재귀성에 쓰인다.

<details><summary>증명</summary>

**(1) 보조정리 (Durrett Lemma 1.17–1.18; Norris §1.8.1).** 기약·비주기이면 모든 $x,y$에 대해 $n_0$이 있어 $n\ge n_0$이면 $p^n(x,y)>0$.

*수론 보조정리.* $I\subset\{1,2,\dots\}$가 덧셈에 닫혀 있고 $\gcd(I)=1$이면 $I$는 충분히 큰 모든 정수를 포함한다. 증명: $\gcd$가 1인 유한 부분집합이 있고 베주(Bezout) 항등식으로 $1=\sum c_is_i$ ($c_i\in\mathbb Z$, $s_i\in I$). 양의 계수 항과 음의 계수 항을 모으면 $1=a-b$, $a,b\in I\cup\{0\}$이고 $b\ge1$이라 해도 된다($b=0$이면 $1\in I$로 끝). $n\ge b^2$을 $n=qb+r$, $0\le r<b\le q$로 쓰면 $n=qb+r(a-b)=(q-r)b+ra\in I$. $\square$

*적용.* $I_x=\{n\ge1:p^n(x,x)>0\}$는 덧셈에 닫혀 있고($p^{m+n}(x,x)\ge p^m(x,x)p^n(x,x)$) $\gcd(I_x)$가 바로 주기 $=1$이므로 $n\ge n_x$이면 $p^n(x,x)>0$. 기약이므로 $p^{k}(x,y)>0$인 $k=k_{xy}$가 있고, $n\ge n_x$이면 $p^{n+k}(x,y)\ge p^n(x,x)p^k(x,y)>0$. 상태쌍이 유한 개이므로 $n_0=\max_{x,y}(n_x+k_{xy})$를 잡는다.

**(2) 곱 사슬.** $Z_n=(X_n,Y_n)$을 $S\times S$ 위에서 **독립**으로 움직이게 하자: 전이확률 $\bar p((x,y),(x',y'))=p(x,x')p(y,y')$. $\bar p^n((x,y),(x',y'))=p^n(x,x')p^n(y,y')$이고 (1)에 의해 $n\ge n_0$이면 양수이므로 곱 사슬은 **기약**이다. $\pi\otimes\pi$는 곱 사슬의 정상분포다: $\sum_{x,y}\pi(x)\pi(y)p(x,x')p(y,y')=\pi(x')\pi(y')$.

**(3) 만남은 확실하다.** 유한 기약 사슬은 재귀(01b)이므로 어떤 상태에서 출발하든 임의의 고정 상태 $(z,z)$에 확률 1로 도달한다. 따라서 대각선 도달 시각 $T_c=\min\{n:X_n=Y_n\}<\infty$ a.s.

**(4) 붙이기.** $W_n=Y_n$ ($n<T_c$), $W_n=X_n$ ($n\ge T_c$)로 정의한다. $T_c$는 $(X,Y)$에 대한 정지시각이고 $X_{T_c}=Y_{T_c}$이므로, 강마르코프 성질에 의해 $T_c$ 이후 $X$의 미래는 $X_{T_c}$에서 출발한 $P$-사슬이며 과거와 독립 — $Y$의 미래와 같은 법칙이다. 따라서 $W$는 초기분포 $\pi$의 $P$-사슬이고, 특히 $W_n\sim\pi$ for all $n$. 구성상 $n\ge T_c$이면 $X_n=W_n$이다.

**(5) 마무리.** $(X_n,W_n)$에 정리 1을 적용하면 $\|P^n(x,\cdot)-\pi\|_{TV}\le P(T_c>n)\xrightarrow[n\to\infty]{}0$ ((3)에 의해). $x$가 유한 개이므로 $d(n)=\max_x\|P^n(x,\cdot)-\pi\|_{TV}\to0$. 유일성은 01d 정리 1(또는 $\nu P^n\to\pi$가 모든 $\nu$에 대해 성립하므로 다른 정상분포 $\nu$는 $\nu=\nu P^n\to\pi$). $\square$

</details>

### 2.4 정리 3 — 가역 사슬의 스펙트럼 상계 (Lawler 2e §1.2)

> 가역 기약 사슬에서
> $$\|P^n(x,\cdot)-\pi\|_{TV}\le\frac{\lambda_*^n}{2\sqrt{\pi(x)}},\qquad\text{따라서}\qquad d(n)\le\frac{\lambda_*^n}{2\sqrt{\pi_{\min}}};$$
> 또한 $\lim_n d(n)^{1/n}=\lambda_*$ (하계도 같은 지수), 즉 $\log d(n)$의 점근 기울기는 $\log\lambda_*$.

**가정이 왜 필요한가**
- **가역성**: $D=\mathrm{diag}(\pi)$에 대해 $D^{1/2}PD^{-1/2}$가 대칭행렬이 되어 실수 고유값과 $\pi$-직교 고유벡터 전개 $p^n(x,y)/\pi(y)=1+\sum_{j\ge2}\lambda_j^nf_j(x)f_j(y)$를 쓸 수 있다. 비가역 사슬은 복소 고유값·조르당 블록이 가능해 상수 $C$가 커지거나 $n^k|\lambda|^n$ 같은 다항 인자가 붙는다(지수는 여전히 $\max_{j\ge2}|\lambda_j|$).
- **$\lambda_*<1$**: 기약(⇒ 고유값 1이 단순, $\lambda_2<1$)과 비주기(⇒ $\lambda_{\min}>-1$)가 필요. 반사 걷기·Ehrenfest는 $\lambda_{\min}=-1$이라 $\lambda_*=1$, 상계가 아무것도 말해 주지 않는다.

**증명 스케치.** $\langle f,g\rangle_\pi=\sum_x\pi(x)f(x)g(x)$에 대해 가역 $P$는 자기수반이므로 정규직교 고유함수 $f_1\equiv1,f_2,\dots$가 있고 $p^n(x,y)/\pi(y)=\sum_j\lambda_j^nf_j(x)f_j(y)$. 코시–슈바르츠로
$$2\|P^n(x,\cdot)-\pi\|_{TV}=\sum_y\pi(y)\Big|\frac{p^n(x,y)}{\pi(y)}-1\Big|\le\Big(\sum_y\pi(y)\Big(\frac{p^n(x,y)}{\pi(y)}-1\Big)^2\Big)^{1/2}=\Big(\sum_{j\ge2}\lambda_j^{2n}f_j(x)^2\Big)^{1/2}\le\lambda_*^n\Big(\sum_jf_j(x)^2\Big)^{1/2}=\frac{\lambda_*^n}{\sqrt{\pi(x)}}.$$
마지막 등식은 정규직교 기저의 완비성 $\sum_jf_j(x)^2=1/\pi(x)$ ($\delta_x/\pi(x)$를 전개). 하계: $|\lambda_j|=\lambda_*$인 고유함수 $f_j$ 방향 성분은 정확히 $|\lambda_j|^n=\lambda_*^n$으로 줄어드는데 TV는 이 성분을 상수배 이상으로 감지하므로 $d(n)\ge c\,\lambda_*^n$. 2-상태 사슬에서는 직접 계산으로 $\|P^n(0,\cdot)-\pi\|_{TV}=\pi(1)|\lambda_2|^n$ — 상계와 **같은 지수**, 다른 상수. $\square$(스케치)

### 2.5 정리 4 — 주기적 사슬의 세자로 수렴 (Durrett 3e §1.5; Norris §1.10; 증명은 04c)

> 기약 유한 사슬(주기 무관)에서 $\frac1n\sum_{k=1}^np^k(x,y)\to\pi(y)$.

**가정이 왜 필요한가**
- **기약**: 극한이 출발점에 무관하려면. **비주기는 불필요** — 진동하는 항들이 평균 속에서 상쇄된다(고유값 $|\lambda|=1$, $\lambda\ne1$인 항의 부분합 $\sum_{k\le n}\lambda^k$는 유계).

### 2.6 직관

두 사람이 **같은 동전**을 공유하며 사슬을 걷는다고 하자. 한 명은 $x$에서, 다른 한 명은 이미 평형 $\pi$에서 출발한다. 둘이 한 번이라도 만나면 그 뒤로는 구별할 수 없으므로, $x$에서 출발한 사슬이 평형과 다른 정도는 "아직 못 만났을 확률"로 눌린다 — 이것이 결합 부등식이다. 만남이 확실하려면 기약(어디든 갈 수 있음)과 비주기(짝수·홀수 위상이 어긋나지 않음)가 필요하다.

속도는 고유값이 정한다. $P^n$을 고유벡터로 전개하면 $\pi$ 성분은 $1^n$으로 남고 나머지는 $\lambda_j^n$으로 죽는다. 그래서 $\log d(n)$은 결국 기울기 $\log\lambda_*$의 직선이다. 날씨 사슬은 $\lambda_2=0.3$이라 5걸음이면 $10^{-3}$, 게으른 Ehrenfest(20)는 $\lambda_2=0.95$라 혼합에 39걸음이 든다.

### 2.7 함정(traps)

1. **경험분포로 TV 거리를 추정하면 항상 양의 편향**이 있다: 노이즈가 절댓값으로 더해지므로 참값이 0이어도 $\approx\sqrt{|S|/N}$ 수준의 바닥이 남는다. 작은 TV 값은 행렬 거듭제곱으로 정확히 계산하고, MC는 단일 확률 $p^n(x,y)$에만 쓴다.
2. $d(n)$의 스펙트럼 상계 상수 $1/(2\sqrt{\pi_{\min}})$는 매우 느슨할 수 있다(Ehrenfest(20): $512$). **지수(기울기)는 정확하지만 절편은 아니다.**
3. $\lambda_*$는 $|\lambda_2|$가 아니라 $\max(|\lambda_2|,|\lambda_{\min}|)$. 비게으른 Ehrenfest는 $\lambda_2=0.9$이지만 $\lambda_{\min}=-1$이라 아예 수렴하지 않는다.
4. 혼합시간은 **최악의 출발점** 기준($\max_x$). Ehrenfest에서 최악은 끝 상태 0(또는 20)이다; 중앙 10에서 출발하면 훨씬 빠르다.
5. 수렴 정리는 **유한** 상태 가정을 쓴다. 무한 사슬에서는 양재귀가 추가로 필요 — 01x의 대칭 랜덤 워크는 기약·재귀·비주기(게으르게 하면)여도 $p^n(0,0)\to0$.

✍️ **내 말로**: (책을 덮고 여기에 핵심 정의·정리를 다시 써 보세요) — (1) TV 거리의 두 정의가 같은 이유, (2) 결합 부등식 한 줄 증명, (3) 수렴 정리에서 비주기성이 쓰이는 정확한 자리를 각각 두 문장으로.

## 3. Predict — 코드를 돌리기 전에 예측

아래 셀의 `None`을 숫자로 바꿔 보세요. 날씨 사슬은 01a의 $P=\begin{pmatrix}0.7&0.3\\0.4&0.6\end{pmatrix}$ (0=맑음, 1=비), $\pi=(4/7,3/7)$, $\lambda_2=0.3$이고, 2-상태 사슬에서는 $p^n(0,0)=\pi(0)+\pi(1)\lambda_2^n$ (01a E2)이 손 계산의 열쇠입니다. 결합 문제는 2.6의 "같은 동전" 그림으로, 고유값 둘은 2.1의 게으른 사슬 공식으로 맞힐 수 있습니다. 혼합시간과 $p^{40}(0,10)$은 감으로 — 5절의 비교표에서 이 값이 '내 예측' 열에 들어갑니다.

In [ ]:
predictions = {
    # 날씨 사슬(lambda_2 = 0.3)에서 맑음(0)에서 출발해 5걸음 뒤 분포와 pi = (4/7, 3/7)의 TV 거리는?
    "tv_weather_5": None,
    # 같은 사슬에서 p^5(0, 0) (5일 뒤에도 맑을 확률)은?
    "p5_sunny_sunny": None,
    # 같은 균등난수 u를 공유하는 결합(u <= 0.4 이면 둘 다 0, u > 0.7 이면 둘 다 1)에서
    # 0과 1에서 출발한 두 사슬이 2걸음 뒤에도 아직 못 만났을 확률 P(T_c > 2)는?
    "p_not_coupled_2": None,
    # 12-원(Z_12) 위 게으른 랜덤 워크(머묾 1/2, 좌우 1/4씩)의 두 번째 고유값 lambda_2는?
    "lambda2_lazy_cycle12": None,
    # 게으른 Ehrenfest(20) P_lazy = (I + P)/2 의 lambda_2는?  (Ehrenfest 고유값은 1 - 2k/n, k = 0..n)
    "lambda2_lazy_ehrenfest20": None,
    # 게으른 Ehrenfest(20)의 혼합시간 t_mix(1/4) (최악 출발점 0 기준, 정수)는?
    # 힌트: 상계 512 * 0.95^n <= 1/4 가 주는 n 과 비교해 보세요.
    "tmix_lazy_ehrenfest20": None,
    # 게으른 Ehrenfest(20)에서 0에서 출발해 40걸음 뒤 상태 10에 있을 확률 p^40(0, 10)은?  (pi(10) = 0.1762)
    "p40_lazy_ehrenfest_0_to_10": None,
}

## 4. Simulate — 시뮬레이션

### 4.1 날씨 사슬: TV 거리는 정확히 $\pi(1)\lambda_2^n$으로 줄어든다

2-상태 사슬은 모든 것이 손으로 풀린다. $p^n(0,0)=\pi(0)+\pi(1)\lambda_2^n$이므로 $\|P^n(0,\cdot)-\pi\|_{TV}=|p^n(0,0)-\pi(0)|=\pi(1)\lambda_2^n$. 먼저 행렬 거듭제곱으로 계산한 정확한 TV가 이 공식과 일치하는지 확인한다.

In [ ]:
P = np.array([[0.7, 0.3], [0.4, 0.6]])           # 01a 날씨 사슬: 0 = Sunny, 1 = Rainy
pi = stationary(P)
lam_w = np.sort(np.linalg.eigvals(P).real)[::-1]  # [1, lambda_2]
n_grid = np.arange(16)
tv_exact = np.array([tv_distance(matrix_power(P, n)[0], pi) for n in n_grid])
tv_formula = pi[1] * lam_w[1] ** n_grid           # pi(1) * lambda_2^n
assert np.allclose(tv_exact, tv_formula)
tv5_exact = float(tv_exact[5])
p5_exact = float(matrix_power(P, 5)[0, 0])
print(f"pi = {pi}, lambda_2 = {lam_w[1]:.4f}")
print("TV(n) for n=1,2,3,5,10:", np.array2string(tv_exact[[1, 2, 3, 5, 10]], precision=7))
print(f"TV(5) = {tv5_exact:.7f},  p^5(0,0) = {p5_exact:.6f}")

이제 몬테카를로. 맑음에서 출발한 `N_PATHS`개 경로의 시각 $n$ 경험분포로 TV를 추정하면 어떻게 될까? 2-상태에서 경험 TV는 $|\hat p_n-p_n|$이고 그 기댓값은 참값이 0이어도 $\approx\sqrt{2/\pi}\cdot\sqrt{p(1-p)/N}\approx\sqrt{1/(2\pi N)}$ — **노이즈 바닥**이다. 반면 개별 확률 $p^5(0,0)$은 편향 없이 추정된다.

In [ ]:
paths_w = simulate_chain(P, 0, int(n_grid[-1]), rng, N_PATHS)
emp_w = np.array([np.bincount(paths_w[:, n], minlength=2) / N_PATHS for n in n_grid])
tv_emp = np.array([tv_distance(emp_w[n], pi) for n in n_grid])
noise_floor = np.sqrt(1 / (2 * np.pi * N_PATHS))
est_p5 = mc_proportion(paths_w[:, 5] == 0)
print(f"MC p^5(0,0) = {est_p5}   exact = {p5_exact:.6f}")
print(f"empirical TV at n = 5, 10, 15: {tv_emp[[5, 10, 15]]}   (noise floor ~ {noise_floor:.4f})")

fig, ax = plt.subplots()
ax.semilogy(n_grid, tv_exact, "o-", label="exact TV (matrix power)")
ax.semilogy(n_grid, tv_formula, "--", label=r"$\pi(1)\,\lambda_2^n = (3/7)\,0.3^n$")
ax.semilogy(n_grid, np.maximum(tv_emp, 1e-6), "s", ms=4, label=f"empirical TV (MC, N={N_PATHS})")
ax.axhline(noise_floor, color="gray", ls=":", label=r"noise floor $\sqrt{1/(2\pi N)}$")
ax.set(xlabel="n (steps)", ylabel="TV distance to $\\pi$", ylim=(1e-6, 1),
       title="Weather chain: TV distance decays like 0.3^n; MC floor shows estimation bias")
ax.legend(loc="lower left");

정확한 TV는 $0.3^n$ 직선 위에 정확히 놓이지만, 경험 TV는 $n\approx5$ 이후 노이즈 바닥에서 멈춘다 — **함정 1**이다. 이 그림 이후로 TV 거리는 항상 행렬 거듭제곱으로 계산한다.

### 4.2 결합: 같은 난수를 공유하는 두 사슬

`simulate_chain`과 같은 규칙(`next = (u > C[x]).sum()`, $C=P$의 행별 누적합)으로 두 사슬을 **같은** $u$로 움직인다. 날씨 사슬의 누적합 행은 $C[0]=(0.7,1)$, $C[1]=(0.4,1)$이므로 $u\le0.4$이면 둘 다 0, $u>0.7$이면 둘 다 1, $u\in(0.4,0.7]$일 때만 서로 다른 곳으로 간다. 즉 매 걸음 만날 확률 $0.7$, 못 만날 확률 $0.3=p(0,0)-p(1,0)=\lambda_2$ — 결합 시각은 기하분포 $P(T_c>n)=0.3^n$이다. 만난 뒤에는 같은 $u$가 같은 다음 상태를 주므로 자동으로 붙어 움직인다.

In [ ]:
# helper (spkit에 없어서 여기서 정의)
def coupled_paths(P, x0, y0, n_steps, rng, n_paths):
    """같은 균등난수 u 로 움직이는 두 사슬 (X, Y) 와 결합 시각 T_c (만나지 못하면 inf).

    x0, y0 는 정수(모든 경로가 거기서 출발) 또는 확률벡터(경로별로 추출, 예: y0 = pi).
    """
    P = np.asarray(P, dtype=float)
    n = P.shape[0]
    C = P.cumsum(axis=1)

    def start(z0):
        if np.ndim(z0) == 0:
            return np.full(n_paths, int(z0))
        return rng.choice(n, size=n_paths, p=np.asarray(z0, dtype=float))

    X = np.empty((n_paths, n_steps + 1), dtype=np.int64)
    Y = np.empty_like(X)
    X[:, 0], Y[:, 0] = start(x0), start(y0)
    for k in range(n_steps):
        u = rng.random(n_paths)                          # 하나의 u 를 두 사슬이 공유
        X[:, k + 1] = np.minimum((u[:, None] > C[X[:, k]]).sum(1), n - 1)
        Y[:, k + 1] = np.minimum((u[:, None] > C[Y[:, k]]).sum(1), n - 1)
    met = X == Y
    T_c = np.where(met.any(axis=1), met.argmax(axis=1), np.inf)
    return X, Y, T_c

In [ ]:
N_STEPS_C = 15
X, Y, T_c = coupled_paths(P, 0, 1, N_STEPS_C, rng, N_PATHS)           # 0 에서, 1 에서 출발
q_apart = float(P[0, 0] - P[1, 0])                                     # 한 걸음 뒤에도 갈라져 있을 확률 = lambda_2
p_not_coupled_2_exact = q_apart ** 2
est_not_coupled_2 = mc_proportion(T_c > 2)
n_c = np.arange(11)
surv_mc = np.array([(T_c > n).mean() for n in n_c])                    # P(T_c > n), MC
surv_exact = q_apart ** n_c
tv_c = pi[1] * lam_w[1] ** n_c                                          # 정확 TV(n)
# 정상 출발: Y_0 ~ pi.  P(T_c > n) = pi(1) * 0.3^n = TV(n) — 이 결합은 등호를 달성한다
X2, Y2, T_c2 = coupled_paths(P, 0, pi, N_STEPS_C, rng, N_PATHS)
surv_stat = np.array([(T_c2 > n).mean() for n in n_c])
print(f"P(T_c > 2): MC {est_not_coupled_2}  exact {p_not_coupled_2_exact:.4f};  TV(2) = {tv_c[2]:.4f}")
print(f"stationary start: P(T_c > 3) MC {surv_stat[3]:.4f} vs TV(3) = {tv_c[3]:.4f}")

fig, ax = plt.subplots()
nz = lambda s: np.where(s > 0, s, np.nan)                               # 0 은 로그축에 못 그리므로 생략
ax.semilogy(n_c, nz(surv_mc), "o", label=r"$P(T_c>n)$, MC ($X_0=0, Y_0=1$)")
ax.semilogy(n_c, surv_exact, "-", label=r"$0.3^n$ (exact)")
ax.semilogy(n_c, nz(surv_stat), "^", ms=5, label=r"$P(T_c>n)$, MC ($Y_0\sim\pi$)")
ax.semilogy(n_c, tv_c, "--", label=r"exact TV$(n)=(3/7)\,0.3^n$")
ax.set(xlabel="n (steps)", ylabel="probability", ylim=(1e-6, 1.5),
       title=r"Coupling inequality: TV(n) $\leq$ P($T_c$ > n)")
ax.legend(loc="lower left");

두 곡선 모두 정확 TV 위에 있다 — 결합 부등식이다. 정상 출발($Y_0\sim\pi$)에서는 $Y_0=0$일 때 $T_c=0$이므로 $P(T_c>n)=\pi(1)\,0.3^n$, 즉 **정확히 TV$(n)$** 이다: 2-상태 사슬에서 이 결합은 최적(등호 달성)이다. 일반 사슬에서는 부등식이 대개 엄격하고, 상계의 좋고 나쁨은 결합을 얼마나 잘 설계했느냐에 달린다.

### 4.3 두 큰 사슬: 스펙트럼 간극이 기울기를 정한다

12-원 위 게으른 랜덤 워크와 게으른 Ehrenfest(20). 둘 다 가역(01f)이고 게으르므로 고유값이 모두 $\ge0$, $\lambda_*=\lambda_2$이다. $d(n)=\max_x\|P^n(x,\cdot)-\pi\|_{TV}$를 반복 곱셈으로 정확히 계산하고, 혼합시간·상계·$\log d(n)$의 기울기를 잰다.

In [ ]:
# helper (spkit에 없어서 여기서 정의)
def lazy_cycle_matrix(m):
    """Z_m 위 게으른 랜덤 워크: 머묾 1/2, 좌우 1/4."""
    P = 0.5 * np.eye(m)
    idx = np.arange(m)
    P[idx, (idx + 1) % m] += 0.25
    P[idx, (idx - 1) % m] += 0.25
    return P

def reflecting_walk_matrix(N):
    """{0..N} 반사 랜덤 워크 (01b/01d): 안쪽 ±1 확률 1/2, p(0,1) = p(N,N-1) = 1."""
    P = np.zeros((N + 1, N + 1))
    i = np.arange(1, N)
    P[i, i - 1] = P[i, i + 1] = 0.5
    P[0, 1] = P[N, N - 1] = 1.0
    return P

def d_of_n(P, pi, n_max):
    """정확한 d(n) = max_x TV(P^n(x,·), pi), n = 0..n_max (반복 곱셈)."""
    d = np.empty(n_max + 1)
    Pn = np.eye(len(P))
    for n in range(n_max + 1):
        d[n] = max(tv_distance(Pn[x], pi) for x in range(len(P)))
        Pn = Pn @ P
    return d

def t_mix(d, eps=0.25):
    """d[n] <= eps 가 처음 성립하는 n."""
    return int(np.argmax(d <= eps))

def log_slope(d, n_lo, n_hi):
    """n in [n_lo, n_hi] 에서 log d(n) 의 최소제곱 기울기."""
    n = np.arange(n_lo, n_hi + 1)
    return float(np.polyfit(n, np.log(d[n_lo:n_hi + 1]), 1)[0])

In [ ]:
N_MAX = 200
P_cyc = lazy_cycle_matrix(12)
P_le = 0.5 * (np.eye(21) + ehrenfest_matrix(20))
pi_cyc, pi_le = stationary(P_cyc), stationary(P_le)
lam_cyc = np.sort(np.linalg.eigvals(P_cyc).real)[::-1]
lam_le = np.sort(np.linalg.eigvals(P_le).real)[::-1]
lam2_cyc_exact = 0.5 + 0.5 * np.cos(2 * np.pi / 12)          # E2: (1 + cos(2 pi k/m))/2, k = 1
lam2_le_exact = 0.5 * (1 + (1 - 2 / 20))                      # Ehrenfest 고유값 1 - 2k/n 을 게으르게
assert np.isclose(lam_cyc[1], lam2_cyc_exact) and np.isclose(lam_le[1], lam2_le_exact)

d_cyc, d_le = d_of_n(P_cyc, pi_cyc, N_MAX), d_of_n(P_le, pi_le, N_MAX)
t_mix_cyc, t_mix_le = t_mix(d_cyc), t_mix(d_le)
assert (t_mix_cyc, t_mix_le) == (14, 39)                       # 정확한 d(n)에서 읽은 결정적 값
C_cyc, C_le = 1 / (2 * np.sqrt(pi_cyc.min())), 1 / (2 * np.sqrt(pi_le.min()))
n_bound_le = np.log(0.25 / C_le) / np.log(lam_le[1])          # 상계 C lambda_2^n <= 1/4 가 주는 n
slope_cyc, slope_le = log_slope(d_cyc, 60, 100), log_slope(d_le, 100, 140)
assert abs(slope_le - np.log(lam_le[1])) < 0.002 and abs(slope_cyc - np.log(lam_cyc[1])) < 0.002

print(f"lazy cycle Z_12 : eig top-4 {np.round(lam_cyc[:4], 6)}, min {lam_cyc[-1]:.2e}; "
      f"d(13)={d_cyc[13]:.4f}, d(14)={d_cyc[14]:.4f} -> t_mix={t_mix_cyc}; C={C_cyc:.2f}")
print(f"lazy Ehrenfest  : eig top-4 {np.round(lam_le[:4], 6)}, min {lam_le[-1]:.2e}; "
      f"d(38)={d_le[38]:.4f}, d(39)={d_le[39]:.4f} -> t_mix={t_mix_le}; C={C_le:.0f}, bound n>={n_bound_le:.1f}")
print(f"slope of log d(n): cycle {slope_cyc:.6f} vs log lambda_2 {np.log(lam_cyc[1]):.6f}; "
      f"Ehrenfest {slope_le:.6f} vs {np.log(lam_le[1]):.6f}")

In [ ]:
n_all = np.arange(N_MAX + 1)
fig, ax = plt.subplots()
ax.semilogy(n_all, d_cyc, color="C0", label=r"lazy cycle $\mathbb{Z}_{12}$: exact $d(n)$")
ax.semilogy(n_all, C_cyc * lam_cyc[1] ** n_all, color="C0", ls="--", label=rf"bound ${C_cyc:.2f}\cdot{lam_cyc[1]:.3f}^n$")
ax.semilogy(n_all, d_le, color="C1", label="lazy Ehrenfest(20): exact $d(n)$")
ax.semilogy(n_all, C_le * lam_le[1] ** n_all, color="C1", ls="--", label=rf"bound ${C_le:.0f}\cdot{lam_le[1]:.2f}^n$")
ax.axhline(0.25, color="gray", ls=":", label=r"$\varepsilon = 1/4$")
for t, c, ha in ((t_mix_cyc, "C0", "right"), (t_mix_le, "C1", "left")):
    ax.axvline(t, color=c, ls=":", alpha=0.7)
    ax.annotate(f"t_mix={t}", (t, 0.4), xytext=(-4 if ha == "right" else 4, 0),
                textcoords="offset points", ha=ha, color=c, fontsize=9)
ax.set(xlabel="n (steps)", ylabel="d(n)", ylim=(1e-7, 1e3),
       title=f"Spectral gap sets the slope: lambda2 = {lam_cyc[1]:.3f} (cycle), {lam_le[1]:.2f} (Ehrenfest)")
ax.legend(loc="lower left", fontsize=8);

두 $d(n)$ 곡선 모두 큰 $n$에서 상계와 **평행**하다(기울기 $=\log\lambda_2$: 함정 2의 "지수는 정확"). 그러나 Ehrenfest의 상계 $512\cdot0.95^n$은 절편이 $512$배 느슨해서($d(0)=1$인데 상계는 $512$) $t_{\rm mix}$를 $148.6$으로 예측하지만 실제는 $39$ — 약 4배 차이다. 원 걷기는 $\pi_{\min}=1/12$라 상수 $\sqrt3\approx1.73$으로 훨씬 타이트하다.

### 4.4 MC 검증: 게으른 Ehrenfest(20), 구석에서 출발

$d(n)$ 자체는 행렬로 계산했으니, MC로는 **개별 확률** $p^{40}(0,10)$을 검증하고(함정 1), 경험분포가 $\pi=\text{Bin}(20,1/2)$로 다가가는 모습을 본다.

In [ ]:
N_STEPS_LE = 150
paths_le = simulate_chain(P_le, 0, N_STEPS_LE, rng, N_PATHS)
p40_exact = float(matrix_power(P_le, 40)[0, 10])
est_p40 = mc_proportion(paths_le[:, 40] == 10)
print(f"p^40(0,10): MC {est_p40}  exact {p40_exact:.6f}  (pi(10) = {pi_le[10]:.4f})")
for n in (10, 40, 150):
    print(f"  exact TV(P^{n}(0,.), pi) = {tv_distance(matrix_power(P_le, n)[0], pi_le):.5f}")

states = np.arange(21)
fig, ax = plt.subplots()
width = 0.27
for j, n in enumerate((10, 40, 150)):
    emp = np.bincount(paths_le[:, n], minlength=21) / N_PATHS
    ax.bar(states + (j - 1) * width, emp, width, label=f"empirical, n = {n}")
ax.stem(states, pi_le, linefmt="k-", markerfmt="ko", basefmt=" ", label=r"$\pi$ = Bin(20, 1/2)")
ax.set(xlabel="state (balls in urn A)", ylabel="probability", xticks=states[::2],
       title="Lazy Ehrenfest(20): approach to equilibrium from the corner")
ax.legend();

$n=10$에서는 질량이 아직 왼쪽에 몰려 있고($d(10)$이 크다), $n=40\approx t_{\rm mix}$에서 거의 겹치며, $n=150$에서는 정확 TV가 $10^{-3}$ 아래라 막대와 점이 MC 노이즈 안에서 구별되지 않는다.

### 4.5 주기적 반례: $P^n$은 진동하고, 세자로 평균은 수렴한다

반사 걷기 $\{0,\dots,4\}$와 **비게으른** Ehrenfest(20)는 주기 2이다. $\|P^n(0,\cdot)-\pi\|_{TV}$와 세자로 평균 $\bar P_n=\frac1n\sum_{k=1}^nP^k$의 TV, 그리고 게으른 버전을 비교한다.

In [ ]:
def tv_and_cesaro(P, pi, n_max):
    """n = 0..n_max 에 대해 TV(P^n(0,·), pi) 와 TV(세자로 평균 (1/n) sum_{k<=n} P^k (0,·), pi)."""
    Pn, S = np.eye(len(P)), np.zeros(len(P))
    tv, tv_ces = np.empty(n_max + 1), np.empty(n_max + 1)
    tv[0] = tv_ces[0] = tv_distance(Pn[0], pi)
    for n in range(1, n_max + 1):
        Pn = Pn @ P
        S += Pn[0]
        tv[n], tv_ces[n] = tv_distance(Pn[0], pi), tv_distance(S / n, pi)
    return tv, tv_ces

P_ref = reflecting_walk_matrix(4)
pi_ref = stationary(P_ref)
P_ref_lazy = 0.5 * (np.eye(5) + P_ref)
P_eh = ehrenfest_matrix(20)                       # 비게으른 Ehrenfest: pi 는 게으른 것과 같다
tv_ref, ces_ref = tv_and_cesaro(P_ref, pi_ref, N_MAX)
tv_ref_lazy, _ = tv_and_cesaro(P_ref_lazy, pi_ref, N_MAX)
tv_eh, ces_eh = tv_and_cesaro(P_eh, pi_le, N_MAX)
assert np.isclose(tv_ref[100], 0.5) and np.isclose(tv_ref[101], 0.5)
print("periods:", classify_states(P_ref)["period"], classify_states(P_eh)["period"], classify_states(P_ref_lazy)["period"])
print(f"reflecting walk : TV(100)={tv_ref[100]:.4f}, TV(101)={tv_ref[101]:.4f}, Cesaro(200)={ces_ref[200]:.4f}, lazy TV(200)={tv_ref_lazy[200]:.2e}")
print(f"Ehrenfest(20)   : TV(200)={tv_eh[200]:.4f}, TV(201)={tv_distance(matrix_power(P_eh, 201)[0], pi_le):.4f}, Cesaro(200)={ces_eh[200]:.4f}")
print("eigenvalues (reflecting):", np.round(np.sort(np.linalg.eigvals(P_ref).real), 4))
print("eigenvalues (Ehrenfest) : min", np.sort(np.linalg.eigvals(P_eh).real)[:3].round(4), "... max 1")

In [ ]:
n_show = np.arange(61)
fig, ax = plt.subplots()
ax.plot(n_show, tv_ref[:61], "-", label="periodic reflecting walk: TV($P^n(0,\\cdot)$, $\\pi$)")
ax.plot(n_show, ces_ref[:61], "--", label="its Cesaro average $\\bar P_n$")
ax.plot(n_show, tv_ref_lazy[:61], "-.", label="lazy reflecting walk: TV($P^n_{lazy}(0,\\cdot)$, $\\pi$)")
ax.plot(n_show, tv_eh[:61], ":", color="C3", label="periodic Ehrenfest(20): TV")
ax.set(xlabel="n (steps)", ylabel="TV distance", ylim=(0, 1.02),
       title="Periodic chain: P^n does not converge, Cesaro average does")
ax.legend(fontsize=8);

반사 걷기의 TV는 처음 몇 걸음 뒤 정확히 $\frac12$에 고정된다: 짝수 시각에는 짝수 상태에만 질량이 있으므로 $\pi$의 홀수 질량 $\frac12$가 통째로 어긋난다. 고유값 목록에 $-1$이 있는 것이 그 원인($|\lambda|=1$인 고유값이 둘, $\lambda_*=1$). 세자로 평균은 $O(1/n)$으로, 게으른 버전은 기하급수적으로 0에 간다.

## 5. Compare — 예측 · 시뮬레이션 · 닫힌 형식 비교

닫힌 형식은 모두 코드로 계산했다: 날씨 사슬은 `matrix_power`와 $\pi(1)\lambda_2^5$, 결합은 $(p(0,0)-p(1,0))^2$, 고유값은 `np.linalg.eigvals`와 2.1의 게으른 사슬 공식, 혼합시간은 정확한 $d(n)$에서 읽은 정수. MC 추정이 있는 행은 셋(개별 확률만 — 함정 1)이고, 나머지는 SE $=0$의 정확 비교다.

In [ ]:
rows = [
    {"name": "p5_sunny_sunny = p^5(0,0)", "predicted": predictions["p5_sunny_sunny"], "estimate": est_p5, "exact": p5_exact},
    {"name": "p_not_coupled_2 = P(T_c > 2)", "predicted": predictions["p_not_coupled_2"], "estimate": est_not_coupled_2, "exact": p_not_coupled_2_exact},
    {"name": "p40_lazy_ehrenfest_0_to_10", "predicted": predictions["p40_lazy_ehrenfest_0_to_10"], "estimate": est_p40, "exact": p40_exact},
    {"name": "tv_weather_5 (not MC)", "predicted": predictions["tv_weather_5"], "estimate": tv5_exact, "exact": float(pi[1] * lam_w[1] ** 5), "se": 0.0},
    {"name": "lambda2_lazy_cycle12 (not MC)", "predicted": predictions["lambda2_lazy_cycle12"], "estimate": float(lam_cyc[1]), "exact": float(lam2_cyc_exact), "se": 0.0},
    {"name": "lambda2_lazy_ehrenfest20 (not MC)", "predicted": predictions["lambda2_lazy_ehrenfest20"], "estimate": float(lam_le[1]), "exact": float(lam2_le_exact), "se": 0.0},
    {"name": "tmix_lazy_ehrenfest20 (exact d(n))", "predicted": predictions["tmix_lazy_ehrenfest20"], "estimate": float(t_mix_le), "exact": None},
]
print(f"exact: p5={p5_exact:.6f}, P(T_c>2)={p_not_coupled_2_exact:.4f}, p40={p40_exact:.6f}, TV(5)={tv5_exact:.7f}, "
      f"lambda2: cycle {lam2_cyc_exact:.6f}, Ehrenfest {lam2_le_exact:.4f}; t_mix: cycle {t_mix_cyc}, Ehrenfest {t_mix_le}")
Markdown(compare_table(rows))

행별로 '불일치'가 뜻하는 것:
- **p5_sunny_sunny**: 어긋나면 `simulate_chain`의 시각 인덱스(`paths[:, 5]`는 5걸음 뒤)가 틀렸거나 $p^n(0,0)=\pi(0)+\pi(1)\lambda_2^n$의 부호를 잘못 둔 것 — 2-상태 사슬의 스펙트럼 전개가 틀릴 수는 없다.
- **p_not_coupled_2**: 어긋나면 결합 구성이 잘못된 것 — 두 사슬이 서로 다른 $u$를 썼거나(그러면 독립 결합, $P(T_c>2)\ne0.09$) 만난 뒤 다시 갈라졌다.
- **p40_lazy_ehrenfest_0_to_10**: 어긋나면 게으른 행렬 $\frac12(I+P)$가 아닌 $P$를 시뮬레이션했거나(홀짝 위상으로 $p^{40}(0,10)$이 달라짐) 상태 인덱스 오프셋.
- **tv_weather_5**: SE $=0$의 정확 비교. 예측이 어긋났다면 계수 $\frac12$(Durrett 표기와 2배 차이) 또는 $\pi(1)$ 대신 $\pi(0)$을 곱한 것.
- **lambda2 두 행**: 정확 비교. `eigvals`의 정렬(실수부 내림차순)과 $\lambda\mapsto\frac12(1+\lambda)$를 확인.
- **tmix_lazy_ehrenfest20**: 닫힌 형식 칸이 비어 있다 — 정확한 $d(n)$에서 읽은 정수 $39$이며, 상계가 주는 $148.6$과의 차이가 함정 2다.

In [ ]:
assert_close(est_p5, p5_exact, k=4, name="p5_sunny_sunny")
assert_close(est_not_coupled_2, p_not_coupled_2_exact, k=4, name="p_not_coupled_2")
assert_close(est_p40, p40_exact, k=4, name="p40_lazy_ehrenfest_0_to_10")
print("all checks passed")

## 6. 연습문제

### E1 계산

2-상태 사슬 $P=\begin{pmatrix}1-a&a\\b&1-b\end{pmatrix}$ ($0\le a,b\le1$)의 고유값, 정상분포($a+b>0$일 때), 그리고 $\|P^n(0,\cdot)-\pi\|_{TV}$의 닫힌 식을 구하시오. $a=0.1$, $b=0.2$, $n=4$에서 값은? 이 사슬이 $\pi$로 수렴하지 **않는** $(a,b)$는 무엇이고, 수렴 정리의 어느 가정이 깨지는가?

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

$\operatorname{tr}P=2-a-b$, $\det P=1-a-b$이므로 고유값은 $1$과 $\lambda_2=1-a-b$. 정상분포는 $\pi(0)(1-a)+\pi(1)b=\pi(0)\Rightarrow a\pi(0)=b\pi(1)$, 즉 $\pi=(b,a)/(a+b)$. 01a E2의 $p^n(0,0)=\pi(0)+\pi(1)\lambda_2^n$에서
$$\|P^n(0,\cdot)-\pi\|_{TV}=|p^n(0,0)-\pi(0)|=\frac{a}{a+b}\,|1-a-b|^n.$$
$a=0.1,b=0.2$: $\lambda_2=0.7$, $\pi=(2/3,1/3)$, TV$(4)=\frac13(0.7)^4=0.080033$.

수렴 실패: $|\lambda_2|=1$인 경우. (i) $a=b=1$: $\lambda_2=-1$, $P=\begin{pmatrix}0&1\\1&0\end{pmatrix}$가 주기 2 — **비주기** 가정 위반, $P^n(0,\cdot)$이 $(1,0),(0,1)$로 진동하고 TV $=\frac12$ 고정. (ii) $a=b=0$: $P=I$, $\lambda_2=1$ — **기약** 가정 위반(정상분포가 무한히 많고 $P^n(0,\cdot)=(1,0)$). $a=0$ 하나만 0이면 $\lambda_2=1-b\in[0,1)$이라 수렴은 하지만 상태 0이 흡수 상태이고 $\pi=(1,0)$.

확인 코드:
```python
a, b = 0.1, 0.2
P2 = np.array([[1 - a, a], [b, 1 - b]])
pi2 = np.array([b, a]) / (a + b)
print(np.linalg.eigvals(P2).real, pi2)                                   # [1, 0.7], (2/3, 1/3)
print(tv_distance(matrix_power(P2, 4)[0], pi2), a / (a + b) * abs(1 - a - b) ** 4)   # 0.080033 both
P_flip = np.array([[0, 1], [1, 0]])
print([tv_distance(matrix_power(P_flip, n)[0], [0.5, 0.5]) for n in range(4)])       # 0.5 forever
```

</details>

### E2 유도 후 시뮬레이션 검증

$m$-원 $\mathbb Z_m$ 위 게으른 랜덤 워크의 고유값이 $\frac12\big(1+\cos(2\pi k/m)\big)$, $k=0,\dots,m-1$임을 유도하시오(고유벡터 $e^{2\pi ikx/m}$). $m=12$에서 수치 고유값과 비교하고, $\log d(n)$의 기울기($n=60..100$)가 $\log\lambda_2=-0.06934$인지 확인하시오. 시뮬레이션: 0에서 출발한 `N_PATHS`개 경로로 $p^{14}(0,0)$을 추정해 정확값과 4 SE 안에서 비교하시오.

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

순환행렬(circulant)은 푸리에 기저로 대각화된다. $(Pf)(x)=\frac12f(x)+\frac14f(x+1)+\frac14f(x-1)$에 $f_k(x)=e^{i\theta x}$, $\theta=2\pi k/m$을 넣으면
$$(Pf_k)(x)=\Big(\frac12+\frac14e^{i\theta}+\frac14e^{-i\theta}\Big)f_k(x)=\frac12\big(1+\cos\theta\big)f_k(x),$$
즉 $f_k$가 고유값 $\lambda_k=\frac12(1+\cos(2\pi k/m))$의 고유벡터이고 $k=0..m-1$의 $m$개가 모두 독립이라 전체 스펙트럼이다. $k=0$이 $1$, $k=\pm1$이 $\lambda_2=\frac12(1+\cos(\pi/6))=0.933013$(중복도 2), $k=m/2$가 최소값 $0$. 게으르지 않은 원 걷기는 $\cos\theta$라서 $k=m/2$에서 $-1$(주기 2). 측정 기울기 $-0.069336=\log\lambda_2$; $p^{14}(0,0)=0.149449$, $t_{\rm mix}(1/4)=14$.

확인 코드:
```python
m = 12
k = np.arange(m)
lam_formula = np.sort(0.5 * (1 + np.cos(2 * np.pi * k / m)))[::-1]
print(np.allclose(lam_formula, lam_cyc), lam_cyc[:3])                    # True, [1, 0.933013, 0.933013]
print(log_slope(d_cyc, 60, 100), np.log(lam_cyc[1]), t_mix(d_cyc))       # -0.069336, -0.069336, 14
paths_cyc = simulate_chain(P_cyc, 0, 14, rng, N_PATHS)
est14 = mc_proportion(paths_cyc[:, 14] == 0)
exact14 = matrix_power(P_cyc, 14)[0, 0]
print(est14, exact14)                                                    # 0.1494 ± 0.0025 (full) vs 0.149449
assert_close(est14, exact14, k=4, name="p14_cycle_0_0")
```

</details>

### E3 가정을 깨보기

비주기성을 깨자: **비게으른** Ehrenfest(20)에서 0에서 출발해 $p^n(0,10)$을 $n=1..60$에서 계산하시오. 홀수 $n$에서는 무슨 값인가? $\|P^n(0,\cdot)-\pi\|_{TV}$는 무엇으로 수렴(또는 고정)하는가? 고유값 목록에서 그 원인을 찾고, (a) 게으르게 만들기, (b) 짝수 시각만 보기($P^2$의 사슬), (c) 세자로 평균 — 세 가지 처방이 각각 **무엇을 무엇으로** 수렴시키는지 실험하시오.

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

홀수 $n$에서 $p^n(0,10)=0$: 매 걸음 공 수가 $\pm1$ 바뀌므로 $X_n\equiv n\pmod2$(홀짝 위상). 짝수 $n$에서는 $p^n(0,10)\to2\pi(10)=0.3524$로 간다. TV는 큰 $n$에서 $\frac12$에 고정된다 — 짝수 시각엔 짝수 상태에만 질량이 있어 $\pi$의 홀수 질량 $\frac12$가 전부 어긋나고, 짝수 상태 위에서는 $2\pi$로 수렴하므로 그 이상은 어긋나지 않는다. 원인: 고유값 $1-2k/20$에 $-1$이 포함 ($|\lambda|=1$인 고유값이 둘 → $\lambda_*=1$, 스펙트럼 간극 0).

- (a) **게으른 사슬** $\frac12(I+P)$: 고유값 $\lambda\mapsto\frac12(1+\lambda)\in[0,1]$, $\lambda_*=0.95$. $P_{\rm lazy}^n(0,\cdot)\to\pi$, $t_{\rm mix}=39$ (4.3절).
- (b) **$P^2$의 사슬**: 짝수 상태 $\{0,2,\dots,20\}$ 위에서 기약·비주기(자기 루프 $p^2(x,x)>0$). $P^{2n}(0,\cdot)\to2\pi\,\mathbf 1_{\rm even}$, 즉 짝수 상태로 조건부한 정상분포 — TV(짝수 시각 분포, $2\pi|_{\rm even}$) $\to0$.
- (c) **세자로 평균** $\frac1n\sum_{k\le n}P^k(0,\cdot)\to\pi$ 자체로 수렴하지만 속도가 $O(1/n)$으로 느리다 (TV: $n=50$에서 $0.249$, $n=200$에서 $0.063$).

확인 코드:
```python
p_n = np.array([matrix_power(P_eh, n)[0, 10] for n in range(1, 61)])
print("odd n:", p_n[0::2][:4], " even n (tail):", p_n[1::2][-2:].round(4))   # 0, 0, ... ; 0.3524 -> 2 pi(10)
print("TV tail:", tv_eh[57:61].round(4))                                     # 0.5 0.5 0.5 0.5
print("eigenvalues:", np.sort(np.linalg.eigvals(P_eh).real).round(2)[:3])   # -1, -0.9, -0.8, ...
print("(a) lazy t_mix:", t_mix(d_le))                                        # 39
P2 = matrix_power(P_eh, 2)
pi_even = np.where(np.arange(21) % 2 == 0, 2 * pi_le, 0.0)                   # 2 pi on even states
print("(b) TV(P^{2n}(0,.), 2pi|even):", [round(tv_distance(matrix_power(P2, n)[0], pi_even), 5) for n in (5, 25, 50)])
print("(c) Cesaro TV:", ces_eh[[10, 50, 200]].round(4))                      # 0.7987, 0.2491, 0.0626
```

</details>

## 7. 정리

1. TV 거리 $\|\mu-\nu\|_{TV}=\max_A|\mu(A)-\nu(A)|=\frac12\sum|\mu-\nu|$; $d(n)=\max_x\|P^n(x,\cdot)-\pi\|_{TV}$는 비증가; $t_{\rm mix}(1/4)=\min\{n:d(n)\le1/4\}$.
2. 수렴 정리: 기약 + 비주기 (+유한) ⇒ $d(n)\to0$. 증명 = 독립 곱 사슬이 대각선에 확률 1로 도달(비주기성은 곱 사슬의 기약성에 쓰임) + 결합 부등식 TV$(n)\le P(T_c>n)$.
3. 가역 사슬: $d(n)\le\frac{1}{2\sqrt{\pi_{\min}}}\lambda_*^n$, $\log d(n)$의 기울기 $=\log\lambda_*$. 날씨 $\lambda_2=0.3$, 게으른 원(12) $0.933$ ($t_{\rm mix}=14$), 게으른 Ehrenfest(20) $0.95$ ($t_{\rm mix}=39$; 상계는 $148.6$을 주어 느슨).
4. 주기적이면 $|\lambda|=1$인 고유값이 여럿 → $P^n$ 진동(반사 걷기·Ehrenfest: TV $=\frac12$ 고정); 세자로 평균·시간 평균만 수렴하고, 게으르게 만들면 해결된다.
5. MC로는 개별 확률 $p^n(x,y)$를 검증하고(세 항목 모두 4 SE 안), TV 거리 자체는 행렬 거듭제곱으로 계산한다 — 경험 TV는 $\sqrt{1/(2\pi N)}$ 수준의 양의 편향이 있다.

**trap 카드**
- Q: $\lambda_2=0.9$인 사슬은 반드시 $0.9^n$ 속도로 평형에 수렴하는가?
- A: 아니다. 수렴 속도는 $\lambda_*=\max(|\lambda_2|,|\lambda_{\min}|)$가 정한다. 비게으른 Ehrenfest(20)는 $\lambda_2=0.9$이지만 $\lambda_{\min}=-1$이라 아예 수렴하지 않는다(주기 2). 게으르게 만들면 $\lambda_*=0.95$.

**다음 노트북: 01f** — 스펙트럼 상계에 쓰인 '가역성'(세부균형)이 무엇인지, 그리고 원하는 $\pi$를 정상분포로 갖는 사슬을 거꾸로 설계하는 법(Metropolis–Hastings)을 배운다 — 오늘 잰 수렴 속도 $\lambda_2$가 MCMC의 효율을 결정한다.

log/progress.md 한 줄 템플릿:
`- YYYY-MM-DD 01e 수렴·혼합 | 소요 __분 | 예측 적중 _/7 | 막힌 곳: ___ | 다음: 01f`